Data Analysis Simple Agent

In [1]:
import os
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.agents import AgentState
from langchain_google_genai import ChatGoogleGenerativeAI
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from datetime import datetime, timedelta

load_dotenv()

np.random.seed(42)

Generate Sample Data

In [2]:
n_rows = 1000

start_date = datetime(2023, 1, 1)
dates = [start_date + timedelta(days=i) for i in range(n_rows)]

# Define data categories
makes = ['Toyota', 'Honda', 'Ford', 'Chevrolet', 'Nissan', 'BMW', 'Mercedes', 'Audi', 'Hyundai', 'Kia']
models = ['Sedan', 'SUV', 'Truck', 'Hatchback', 'Coupe', 'Van']
colors = ['Red', 'Blue', 'Black', 'White', 'Silver', 'Gray', 'Green']

# Create the dataset
data = {
    'Date': dates,
    'Make': np.random.choice(makes, n_rows),
    'Model': np.random.choice(models, n_rows),
    'Color': np.random.choice(colors, n_rows),
    'Year': np.random.randint(2015, 2023, n_rows),
    'Price': np.random.uniform(20000, 80000, n_rows).round(2),
    'Mileage': np.random.uniform(0, 100000, n_rows).round(0),
    'EngineSize': np.random.choice([1.6, 2.0, 2.5, 3.0, 3.5, 4.0], n_rows),
    'FuelEfficiency': np.random.uniform(20, 40, n_rows).round(1),
    'SalesPerson': np.random.choice(['Alice', 'Bob', 'Charlie', 'David', 'Eva'], n_rows)
}

# Create DataFrame and sort by date
df = pd.DataFrame(data).sort_values('Date')

print("\nFirst few rows of the generated data:")
print(df.head())

print("\nDataFrame info:")
df.info()

print("\nSummary statistics:")
print(df.describe())




First few rows of the generated data:
        Date       Make      Model  ... EngineSize  FuelEfficiency  SalesPerson
0 2023-01-01   Mercedes      Sedan  ...        2.0            24.7        Alice
1 2023-01-02  Chevrolet  Hatchback  ...        1.6            26.2          Bob
2 2023-01-03       Audi      Truck  ...        2.0            28.0        David
3 2023-01-04     Nissan  Hatchback  ...        3.5            24.8        David
4 2023-01-05   Mercedes  Hatchback  ...        2.5            24.1        Alice

[5 rows x 10 columns]

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            1000 non-null   datetime64[ns]
 1   Make            1000 non-null   object        
 2   Model           1000 non-null   object        
 3   Color           1000 non-null   object        
 4   Year            10

Create Data Analysis Agent

In [7]:
agent = create_pandas_dataframe_agent(
    ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7),
    df,
    verbose=True,
    allow_dangerous_code=True,
)

print("Data Analysis Agent is ready. You can now ask questions about the data.")

Data Analysis Agent is ready. You can now ask questions about the data.


Define Question-Asking Function

In [12]:
def ask_agent(question):
    """Function to ask questions to the agent and display the response."""
    response = agent.invoke({
        "input": question,
        "agent_scratchpad": f"Human: {question}\nAI: To answer this question, I need to use python to analyze the dataframe. I'll use the python_repl_ast tool.\n\nAction: python_repl_ast\nAction Input: "
    })
    print(f"Question: {question}")
    print(f"Answer: {response}")
    print("---")

In [14]:
ask_agent("What are the column names in this dataset?")
ask_agent("How many rows are in this dataset?")
ask_agent("What is the average price of cars sold?")



> Entering new AgentExecutor chain...
Action: python_repl_ast
Action Input: print(df.columns)Index(['Date', 'Make', 'Model', 'Color', 'Year', 'Price', 'Mileage',
       'EngineSize', 'FuelEfficiency', 'SalesPerson'],
      dtype='object')
I now know the final answer
Final Answer: The column names in the dataset are 'Date', 'Make', 'Model', 'Color', 'Year', 'Price', 'Mileage', 'EngineSize', 'FuelEfficiency', and 'SalesPerson'.

> Finished chain.
Question: What are the column names in this dataset?
Answer: {'input': 'What are the column names in this dataset?', 'agent_scratchpad': "Human: What are the column names in this dataset?\nAI: To answer this question, I need to use python to analyze the dataframe. I'll use the python_repl_ast tool.\n\nAction: python_repl_ast\nAction Input: ", 'output': "The column names in the dataset are 'Date', 'Make', 'Model', 'Color', 'Year', 'Price', 'Mileage', 'EngineSize', 'FuelEfficiency', and 'SalesPerson'."}
---


> Entering new AgentExecutor chain..